# The CUDA programming model: threads, blocks, warps and SMs

How one kernel launch becomes thousands of threads, how they are grouped into blocks and warps, how blocks land on streaming multiprocessors, and how to write, launch, check and time your first real kernel: vector add, measured against the bandwidth limit.

This notebook holds the code from the lesson **[The CUDA programming model: threads, blocks, warps and SMs](https://inferenceacademy.com/learn/kern-start-here/the-cuda-programming-model)** on Inference Academy, in the same order. Read the lesson for the explanations, diagrams and exercises, and run the cells here as you go.

**Before you start:** choose *Runtime → Change runtime type → T4 GPU*, then run the first cell to check the GPU.

In [ ]:
!nvidia-smi

## One launch, many threads

## Warps: the unit the hardware actually runs

## Blocks land on SMs

## Your first real kernel: vector add

In [ ]:
%%writefile vadd.cu
// vadd.cu: c = a + b, timed with CUDA events
#include <cstdio>
#include <cstdlib>

#define CHECK(x) do { cudaError_t e = (x); if (e != cudaSuccess) { \
  printf("CUDA error %s at %s:%d\n", cudaGetErrorString(e), __FILE__, __LINE__); exit(1); } } while (0)

__global__ void vadd(const float* a, const float* b, float* c, int n) {
  int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) c[i] = a[i] + b[i];
}

int main() {
  const int n = 1 << 26;                      // 64M floats, 256 MB per array
  const size_t bytes = n * sizeof(float);
  float *a, *b, *c;
  CHECK(cudaMalloc(&a, bytes)); CHECK(cudaMalloc(&b, bytes)); CHECK(cudaMalloc(&c, bytes));
  CHECK(cudaMemset(a, 0, bytes)); CHECK(cudaMemset(b, 0, bytes));

  const int threads = 256;
  const int blocks = (n + threads - 1) / threads;   // round up
  cudaEvent_t start, stop;
  CHECK(cudaEventCreate(&start)); CHECK(cudaEventCreate(&stop));

  vadd<<<blocks, threads>>>(a, b, c, n);       // warm-up
  CHECK(cudaGetLastError());
  CHECK(cudaEventRecord(start));
  const int reps = 20;
  for (int r = 0; r < reps; r++) vadd<<<blocks, threads>>>(a, b, c, n);
  CHECK(cudaEventRecord(stop));
  CHECK(cudaEventSynchronize(stop));

  float ms; CHECK(cudaEventElapsedTime(&ms, start, stop)); ms /= reps;
  double gbs = 3.0 * bytes / (ms / 1e3) / 1e9;
  printf("%.3f ms per call, %.1f GB/s\n", ms, gbs);
}

In [ ]:
!nvcc -O3 -arch=sm_75 -o vadd vadd.cu && ./vadd

### What to expect

### Grid-stride loops

Snippet from the lesson (add it to your program; not run on its own):

```cpp
__global__ void vadd_strided(const float* a, const float* b, float* c, int n) {
  for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < n; i += blockDim.x * gridDim.x)
    c[i] = a[i] + b[i];
}
// launch with, say, 4 blocks per SM: vadd_strided<<<40 * 4, 256>>>(a, b, c, n);
```

## Waves and tails

## The CPU and GPU are separate machines

## Finding bugs

In [ ]:
!compute-sanitizer ./vadd

## What you can now do

## Common mistakes

## Exercise

## Going deeper

---
Code from [Inference Academy](https://inferenceacademy.com), MIT licensed. Lesson: https://inferenceacademy.com/learn/kern-start-here/the-cuda-programming-model